In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import sys
sys.path.insert(0, '/home/codycruz/LLC_ocean_emulator/high_res_diagnostics')
from predictions_io import open_predictions
import cftime
import os

In [2]:
# ============== LOAD LLC ==============

# DOMAIN
# i_extent=slice(2880,3600)
# i_g_extent=slice(2880,3600)
# j_extent=slice(720,1440)
# j_g_extent=slice(720,1440)
i_extent=slice(0,4320)
i_g_extent=slice(0,4320)
j_extent=slice(0,4320)
j_g_extent=slice(0,4320)


llc_patch_full_face = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320',consolidated=False).isel(
    face=1,
    i=i_extent,
    i_g=i_g_extent,
    j=j_extent,
    j_g=j_g_extent
)

In [3]:
# i_extent=slice(2880+100+25,2880+100+25+20)
# i_g_extent=slice(2880+100+25,2880+100+25+20)
# j_extent=slice(720+100+75,720+100+75+20)
# j_g_extent=slice(720+100+75,720+100+75+20)
# i_extent=slice(2880,2880+720)
# i_g_extent=slice(2880,2880+720)
# j_extent=slice(720,720+720)
# j_g_extent=slice(720,720+720)
i_extent=slice(2000,2300)
i_g_extent=slice(2000,2300)
j_extent=slice(2700,3000)
j_g_extent=slice(2700,3000)

llc_patch_full = llc_patch_full_face.isel(
    i=i_extent,
    i_g=i_g_extent,
    j=j_extent,
    j_g=j_g_extent
)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': 'face-1-ckpt-17',
        'key': 'emulator_1',
        'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face//epoch17-ema/predictions.zarr',
        'desc': ''
    },
#     {
#         'name': 'face-1-ckpt4',
#         'key': 'emulator_2',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face/epoch4-ema-quintic-reg/predictions.zarr',
#         'desc': ''
#     },
#     {
#         'name': 'band_h-ckpt4',
#         'key': 'emulator_3',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face//epoch4-ema-quintic-band_h/predictions.zarr',
#         'desc': ''
#     },
#    {
#         'name': 'band_hz-ckpt4',
#         'key': 'emulator_4',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face//epoch4-ema-quintic-band_hz/predictions.zarr',
#         'desc': ''
#     },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    # Load emulator data
    emu_ds = open_predictions(cfg['path'])
    
    # Rename dimensions to match LLC if needed
    # (adjust these names based on your actual emulator dimension names)
    dim_mapping = {}
    if 'lat' in emu_ds.dims and 'j' not in emu_ds.dims:
        dim_mapping['lat'] = 'j'
    if 'lon' in emu_ds.dims and 'i' not in emu_ds.dims:
        dim_mapping['lon'] = 'i'
    
    if dim_mapping:
        emu_ds = emu_ds.rename(dim_mapping)
    
    # Apply same spatial slicing as LLC
    emu_ds = emu_ds.isel(i=i_extent, j=j_extent)
    
    # Align emulator to LLC's coordinates
    emu_ds = emu_ds.assign_coords({
        'i': llc_patch_full.i,
        'j': llc_patch_full.j
    })
    
    emulator_patches_raw[cfg['key']] = emu_ds
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z', 'dxC', 'dyC', 'dxG', 'dyG']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    # Add LLC grid variables
    for gv in grid_vars:
        if gv in llc_patch:
            patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")
    print(f"    Shape: {emulator_patches[key][list(emulator_patches[key].data_vars)[0]].shape}")
    print(f"    i range: [{emulator_patches[key].i.values.min()}, {emulator_patches[key].i.values.max()}]")
    print(f"    j range: [{emulator_patches[key].j.values.min()}, {emulator_patches[key].j.values.max()}]")

print(f"LLC shape: {llc_patch[list(llc_patch.data_vars)[0]].shape}")

Loaded face-1-ckpt-17: 
LLC subset to 744 common times

=== Setup complete: LLC + 1 emulators ===
  face-1-ckpt-17 (emulator_1)
    Shape: (744, 51, 300, 300)
    i range: [2000, 2299]
    j range: [2700, 2999]
LLC shape: (300, 300)


In [4]:
selected_time_range = [0, 744]   # inclusive indices
stepping = 186                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:744
Stepping = 186
Final synchronized length = 4
LLC now has 4 times
face-1-ckpt-17 (emulator_1) now has 4 times


In [ ]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"


In [6]:
# ============== VERTICAL-AXIS HELPERS (W lives on cell interfaces) ==============
# LLC keeps W on `k_p1` -- 52 cell INTERFACES -- while Theta/Salt/U/V sit on `k`,
# 51 cell CENTRES. The emulator writes W back onto `k` with 51 levels. So a
# blanket `.isel(k=...)` silently leaves LLC's W at full depth (the level index
# never applies) and the plot then fails on a 3-D array, or a difference
# broadcasts 52 against 51. Level L means the same layer in both: the emulator's
# `k=L` is trained against LLC's `k_p1=L`.
VERTICAL_DIMS = ("k", "k_p1", "k_l", "k_u", "lev")


def vertical_dim_of(da):
    """Whichever vertical dimension this variable carries, or None if surface."""
    for dim in VERTICAL_DIMS:
        if dim in da.dims:
            return dim
    return None


def pick(patch, var, *, time=None, k=None):
    """`patch[var]` with `time`/`k` applied to the axes it actually has."""
    da = patch[var]
    if time is not None:
        da = da.isel(time=time)
    if k is not None:
        dim = vertical_dim_of(da)
        if dim is not None:
            da = da.isel({dim: k})
    return da


def common_levels(*patch_var_pairs):
    """Levels the given (patch, var) pairs share.

    LLC's W has 52 and the emulator's has 51, so depth comparisons have to trim
    to the shorter of the two rather than broadcast.
    """
    sizes = []
    for patch, var in patch_var_pairs:
        dim = vertical_dim_of(patch[var])
        sizes.append(patch[var].sizes[dim] if dim is not None else 1)
    return min(sizes)


def depth_axis(patch, var):
    """Depth in metres for each level of `var`, negative downward.

    The store carries `Z` for cell centres but no coordinate for the interfaces
    W sits on, so derive them: face[0] = 0, face[i+1] = 2*centre[i] - face[i].
    That is the same construction as DEPTH_INTERFACES in ocean_emulators.
    """
    centres = np.asarray(patch['Z'].values, dtype=float)
    if vertical_dim_of(patch[var]) != 'k_p1':
        return centres
    faces = [0.0]
    for centre in centres:
        faces.append(2.0 * centre - faces[-1])
    return np.asarray(faces)


# surface field and field difference

In [7]:
# ============== SET VARIABLES HERE ==============
# prog_vars = ['Theta', 'W', 'U']#, 'Salt', 'U', 'V']
# colormaps = {'Theta': 'Spectral_r', 'W': 'magma', 'U': 'bwr'}#, 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}

prog_vars = ['Eta']#, 'U','V','Eta']#, 'Salt', 'U', 'V']
colormaps = {'Theta': 'Spectral_r','Salt': 'viridis', 'U': 'bwr', 'V': 'bwr','Eta': 'viridis'}#{'Theta': 'Spectral_r', 'Eta': 'viridis'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/simple/', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields for this row
        fields = [pick(llc_patch, var, time=t, k=0)]
        for emu_name, emu_key in emulator_info:
            fields.append(pick(emulator_patches[emu_key], var, time=t, k=0))
        
        # Calculate vmin/vmax for this row only
        finite_fields = [f.values[np.isfinite(f.values)] for f in fields]
        if not any(vals.size for vals in finite_fields):
            print(f"  Skipping {var} {time_str}: no finite values")
            continue
        vmin = np.min([vals.min() for vals in finite_fields if vals.size])
        vmax = np.max([vals.max() for vals in finite_fields if vals.size])
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        cf = None
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            field_values = np.ma.masked_invalid(field.values)
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field_values,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
        
        # Add one colorbar for this row (shared by both columns)
        cbar = fig.colorbar(cf, ax=axes[row, :], pad=0.02, fraction=0.046)

    plt.savefig(f'figs/simple/fields-{var}.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for Eta...
✓ Saved plots for Eta


# Surface Gradients

In [ ]:
grad_vars = ['Theta']#, 'Salt', 'U', 'V']

metric_vars = {
    'Theta': (None, None),
    'Salt': (None, None),
    'U': ('dxC', 'dyG'),   # U is on (j, i_g)
    'V': ('dxG', 'dyC'),   # V is on (j_g, i)
}


def _metric_values(patch, var):
    # Unknown variables default to tracer metrics, so adding e.g. 'W' to
    # grad_vars needs no edit here -- W is on (j, i) like Theta and Salt.
    dx_name, dy_name = metric_vars.get(var, (None, None))
    if dx_name is None or dy_name is None:
        dx = np.sqrt(patch['rA'].values)  # tracer variables are on (j, i)
        dy = dx
    else:
        dx = patch[dx_name].values
        dy = patch[dy_name].values
    return dx, dy


for var in grad_vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        da_var = patch[var]
        data = da_var.values  # expected: (time, k, y, x)
        dx, dy = _metric_values(patch, var)
        
        if dx.shape != data.shape[-2:] or dy.shape != data.shape[-2:]:
            raise ValueError(
                f"{patch_name} {var}: data horizontal shape {data.shape[-2:]} "
                f"does not match dx {dx.shape} / dy {dy.shape}"
            )
        
        d_di = (np.roll(data, -1, axis=-1) - np.roll(data, 1, axis=-1)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=-2) - np.roll(data, 1, axis=-2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (da_var.dims, grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape} on {da_var.dims[-2:]}")

print("Done computing gradients!")

Computing grad_Theta...


ValueError: conflicting sizes for dimension 'time': length 1 on 'grad_Theta' and length 4 on {'j': 'CS', 'i': 'CS', 'time': 'Eta', 'k': 'PHrefC', 'k_p1': 'PHrefF', 'i_g': 'SIuice', 'j_g': 'SIvice', 'k_l': 'Zl', 'k_u': 'Zu'}

In [ ]:
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in grad_vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(1):
                field = grad_data[t, k]
                threshold = np.nanpercentile(field, 99)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc Theta: 1322065 high-gradient pixels (1.0%)
✓ llc Salt: 1322065 high-gradient pixels (1.0%)
✓ llc U: 1322045 high-gradient pixels (1.0%)
✓ llc V: 1322045 high-gradient pixels (1.0%)
✓ emulator_1 Theta: 1322176 high-gradient pixels (1.0%)
✓ emulator_1 Salt: 1322175 high-gradient pixels (1.0%)
✓ emulator_1 U: 1322175 high-gradient pixels (1.0%)
✓ emulator_1 V: 1322175 high-gradient pixels (1.0%)
Done creating gradient masks!


In [ ]:
drift_colours = {
    'llc_only': '#FC9065',   # orange
    'emu_only': '#BE3977',   # pink
    'overlap':  '#FCFCBE',   # light tan
}

In [ ]:
for var in grad_vars:
    print(f"Generating surface gradient drift figure for {var}...")
    
    os.makedirs(f'figs/simple/', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    # Get coordinate blocks for lat/lon ticks
    def _coord_block(patch, name, n_j, n_i):
        coord = patch[name]
        j_dim = 'j' if 'j' in coord.dims else ('lat' if 'lat' in coord.dims else 'y')
        i_dim = 'i' if 'i' in coord.dims else ('lon' if 'lon' in coord.dims else 'x')
        sel = coord.isel({
            j_dim: slice(0, n_j),
            i_dim: slice(0, n_i),
        })
        for extra_dim in set(sel.dims) - {j_dim, i_dim}:
            sel = sel.isel({extra_dim: 0})
        return sel.transpose(j_dim, i_dim).values
    
    def _nice_lon_lat_ticks(coord_2d, axis_vals, axis):
        idx = np.linspace(0, len(axis_vals) - 1, 5).round().astype(int)
        pos = axis_vals[idx]
        if axis == 'i':
            coord_vals = coord_2d[coord_2d.shape[0] // 2, idx]
        else:
            coord_vals = coord_2d[idx, coord_2d.shape[1] // 2]
        return pos, [f'{v:.1f}' for v in coord_vals]
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        # Get coordinate blocks and ticks
        n_j, n_i = llc_mask_surface.shape
        j_vals = np.arange(n_j)
        i_vals = np.arange(n_i)
        
        xc_block = _coord_block(llc_patch, 'XC', n_j, n_i)
        yc_block = _coord_block(llc_patch, 'YC', n_j, n_i)
        x_tick_pos, x_tick_labels = _nice_lon_lat_ticks(xc_block, i_vals, 'i')
        y_tick_pos, y_tick_labels = _nice_lon_lat_ticks(yc_block, j_vals, 'j')
        
        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            emu_field = pick(all_patches[emu_key], var, time=t, k=0).values
            emu_mask_surface = gradient_masks[emu_key][var][t, 0]
            
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            ax.scatter(llc_i, llc_j, c=drift_colours['llc_only'], s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c=drift_colours['emu_only'], s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c=drift_colours['overlap'], s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            n_overlap = np.sum(overlap_mask)
            pix = int(0.025 * np.size(pick(all_patches['emulator_1'], var, time=t, k=0).lat) * np.size(pick(all_patches['emulator_1'], var, time=t, k=0).lon))
            ax.set_title(f'{emu_name} {var} {time_str} overlap={n_overlap}/{pix}', fontsize=8)
            
            ax.set_xticks(x_tick_pos)
            ax.set_xticklabels(x_tick_labels, fontsize=7)
            ax.set_yticks(y_tick_pos)
            ax.set_yticklabels(y_tick_labels, fontsize=7)
            ax.set_xlabel('Longitude', fontsize=7)
            ax.set_ylabel('Latitude', fontsize=7)
            
            # if col == ncols - 1:
            #     ax.legend(fontsize=5, loc='upper right', markerscale=5)
        
    
    plt.tight_layout()
    plt.savefig(f'figs/simple/gradients-{var}.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for Theta...
✓ Saved gradient drift figure for Theta
Generating gradient drift figure for Salt...
✓ Saved gradient drift figure for Salt
Generating gradient drift figure for U...
✓ Saved gradient drift figure for U
Generating gradient drift figure for V...
✓ Saved gradient drift figure for V
Done with gradient drift figures!
